# Atlanta BeltLine Capstone: Statistical Analysis & Data Preparation

## Research Question
How does proximity to the Atlanta BeltLine relate to housing costs and racial composition in Atlanta?

## Objective
This notebook is the statistical foundation for my capstone project. Before building machine learning models, I wanted to understand what is happening in the data first.

The main focus is on the relationship between BeltLine proximity, housing costs, and racial demographic change. I am especially looking at whether areas closer to the BeltLine show signs of rising costs and shifts in Black and White population percentages.

This notebook includes:
- exploratory data analysis
- descriptive statistics
- visual distributions
- correlation analysis
- formal statistical testing
- ANOVA testing by distance group
- Pearson correlation testing
- OLS regression
- pre-ML data diagnosis
- multicollinearity checks
- class imbalance checks
- missing values and outlier review


# Problem Statement

The Atlanta BeltLine is often discussed as a redevelopment project that brings trails, green space, investment, and new development into the city. But redevelopment does not affect everyone the same way.

For this capstone, I am looking at whether neighborhoods closer to the BeltLine are experiencing higher housing costs and demographic shifts. The larger issue is not just whether the BeltLine brought investment, but who was able to stay and benefit from that investment.

This matters because historically Black neighborhoods in Atlanta have been central to the city’s culture and identity. If redevelopment raises housing costs and changes the racial makeup of nearby neighborhoods, then the data can help show where displacement pressure may be happening.


# Data Definition

This dataset contains census tract-level demographic, housing, and economic data for Atlanta across three time periods: 2000, 2010, and 2020.

Key variables include:
- `median_income`: median household income by tract
- `median_home_value`: median owner-occupied home value
- `median_gross_rent`: median gross rent
- `pct_black`: percent of residents who are Black
- `pct_white`: percent of residents who are White
- `pct_hispanic`: percent of residents who are Hispanic
- `pct_renter`: percent of occupied housing units that are renter-occupied
- `poverty_rate`: percent of population below poverty
- `centroid_distance_miles`: distance from each tract centroid to the BeltLine
- `beltline_half_mile`: whether the tract is within 0.5 miles of the BeltLine
- `beltline_one_mile`: whether the tract is within 1 mile of the BeltLine
- `beltline_three_miles`: whether the tract is within 3 miles of the BeltLine
- `year`: census/ACS time period

The dataset has {df.shape[0]} rows and {df.shape[1]-1} original columns.


In [ ]:
# import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# stats testing
from scipy.stats import ttest_ind, f_oneway, pearsonr

# regression and vif
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# model prep
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# make sure pandas shows all columns
pd.set_option('display.max_columns', None)

# suppress warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# load dataset

df = pd.read_csv('atlanta_spatial_final.csv')

# display first few rows
df.head()


In [ ]:
# check the size of the data

df.shape


In [ ]:
# check column names and data types

df.info()


In [ ]:
# check the first few rows again to make sure everything loaded correctly

df.head(10)


In [ ]:
# create distance groups so I can compare areas around the BeltLine

bins = [0, 0.5, 1, 1.5, 3, 5, 10, np.inf]

labels = [
    '0-0.5 miles',
    '0.5-1 mile',
    '1-1.5 miles',
    '1.5-3 miles',
    '3-5 miles',
    '5-10 miles',
    '10+ miles'
]

df['distance_group'] = pd.cut(
    df['centroid_distance_miles'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df[['tract_id', 'year', 'centroid_distance_miles', 'distance_group']].head()


# 1. Exploratory Data Analysis (EDA)

The first step is to understand the structure of the dataset before jumping into models. I looked at descriptive statistics, distributions, and relationships between key variables.

The variables I focused on are income, home value, rent, race percentages, poverty rate, renter percentage, and distance from the BeltLine.


In [ ]:
# key variables for the analysis

key_vars = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'pct_white',
    'pct_hispanic',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

# descriptive statistics
df[key_vars].describe()


In [ ]:
# calculate mean, median, standard deviation, and iqr

eda_summary = df[key_vars].agg(['mean', 'median', 'std']).T
eda_summary['iqr'] = df[key_vars].quantile(.75) - df[key_vars].quantile(.25)

eda_summary


## EDA Interpretation

Looking at the full dataset, the average median income is $67,223, while the median income is $57,073. The average median home value is $269,061, and the average median gross rent is $1,122.

For race composition, the average tract is 51.1% Black and 33.6% White. This matters because the project is not only about housing costs, but about how those housing costs connect to racial change across Atlanta.

The average distance from a tract centroid to the BeltLine is 6.44 miles, but the median distance is 5.07 miles. That tells me the dataset includes both very close BeltLine neighborhoods and areas much farther away, which is useful for comparing spatial patterns.

The IQR for home value is $244,000, which shows that housing values vary a lot across tracts. That variation is important because it reflects how uneven housing costs are across the city.


In [ ]:
# average values by year

year_summary = df.groupby('year')[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'poverty_rate']
].mean()

year_summary


In [ ]:
# average values within 3 miles of the BeltLine by year

within3_summary = df[df['beltline_three_miles'] == 1].groupby('year')[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'poverty_rate']
].mean()

within3_summary


In [ ]:
# calculate percent change from 2000 to 2020 within 3 miles

within3 = df[df['beltline_three_miles'] == 1]

change_within3 = within3.groupby('year')[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'poverty_rate']
].mean()

percent_change_2000_2020 = (
    (change_within3.loc[2020] - change_within3.loc[2000]) /
    change_within3.loc[2000]
) * 100

percent_change_2000_2020


## Change Over Time Interpretation

Within 3 miles of the BeltLine, the changes from 2000 to 2020 are pretty clear.

Median income increased from $31,979 in 2000 to $71,021 in 2020. That is a 122.1% increase.

Median home value increased from $127,596 to $353,061, which is a 176.7% increase.

Median gross rent increased from $555 to $1,279, which is a 130.7% increase.

The racial shift is also important. The average Black population share within 3 miles decreased from 70.9% in 2000 to 43.9% in 2020. At the same time, the White population share increased from 22.8% to 42.2%.

This is one of the strongest patterns in my capstone. The areas near the BeltLine did not just become more expensive. They also became less Black and more White over time.


In [ ]:
# average values by distance group

distance_summary = df.groupby('distance_group')[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'poverty_rate', 'pct_renter']
].mean()

distance_summary


## Distance Group Interpretation

Looking across distance groups helps show how spatial location matters. The BeltLine is not just a random line on the map. Distance from it helps explain where certain economic and demographic patterns are stronger.

The distance group table helps compare areas within 0.5 miles, 1 mile, 3 miles, and farther away. This gives more detail than only saying “near” or “far.”

One important thing I noticed is that the relationship is not perfectly linear for every variable. That makes sense because Atlanta neighborhoods have their own histories. Some areas farther from the BeltLine are also high income, while some areas close to the BeltLine still show poverty and renter pressure. So the point is not that distance explains everything, but that distance is one important part of the story.


# Visual Distributions

Next, I visualized the distributions for the main variables. This helps me see whether the data is normally distributed, skewed, or affected by outliers.


In [ ]:
# distribution of median income

plt.figure(figsize=(8,5))
sns.histplot(df['median_income'].dropna(), kde=True)
plt.title('Distribution of Median Income')
plt.xlabel('Median Income')
plt.ylabel('Number of Tracts')
plt.show()


In [ ]:
# distribution of median home value

plt.figure(figsize=(8,5))
sns.histplot(df['median_home_value'].dropna(), kde=True)
plt.title('Distribution of Median Home Value')
plt.xlabel('Median Home Value')
plt.ylabel('Number of Tracts')
plt.show()


In [ ]:
# distribution of median gross rent

plt.figure(figsize=(8,5))
sns.histplot(df['median_gross_rent'].dropna(), kde=True)
plt.title('Distribution of Median Gross Rent')
plt.xlabel('Median Gross Rent')
plt.ylabel('Number of Tracts')
plt.show()


In [ ]:
# distribution of pct black

plt.figure(figsize=(8,5))
sns.histplot(df['pct_black'].dropna(), kde=True)
plt.title('Distribution of Percent Black')
plt.xlabel('Percent Black')
plt.ylabel('Number of Tracts')
plt.show()


In [ ]:
# boxplot for rent by distance group

plt.figure(figsize=(12,6))
sns.boxplot(data=df, x='distance_group', y='median_gross_rent')
plt.title('Median Gross Rent by BeltLine Distance Group')
plt.xlabel('Distance from BeltLine')
plt.ylabel('Median Gross Rent')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# boxplot for pct black by distance group

plt.figure(figsize=(12,6))
sns.boxplot(data=df, x='distance_group', y='pct_black')
plt.title('Percent Black by BeltLine Distance Group')
plt.xlabel('Distance from BeltLine')
plt.ylabel('Percent Black')
plt.xticks(rotation=45)
plt.show()


## Distribution Interpretation

The distributions show that housing variables are not perfectly normal. Income, home value, and rent are skewed because some tracts have much higher values than others.

This is expected for Atlanta because neighborhoods do not have the same housing market conditions. Some areas are extremely expensive, while others are still facing high poverty or lower property values.

The boxplots are helpful because they show both the typical value and the outliers. I decided to keep the outliers because they represent real neighborhood differences, not just data errors.


# Correlation Analysis

The correlation matrix helps show how variables move together. This is important before machine learning because highly related features can affect model performance and interpretation.


In [ ]:
# correlation matrix for the main variables

corr_vars = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'pct_white',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

corr_matrix = df[corr_vars].corr()

corr_matrix


In [ ]:
# heatmap of correlations

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='BrBG', center=0, fmt='.2f')
plt.title('Correlation Matrix of Key Variables')
plt.show()


## Correlation Interpretation

The correlation matrix shows that housing and income variables are strongly connected. Median income, median home value, and median gross rent move together, which makes sense because higher income areas usually have higher housing costs.

The Pearson correlation between median income and median gross rent is 0.731, with a p-value of < 0.001. This is a strong and statistically significant relationship.

The correlation between percent Black and median income is -0.691, with a p-value of < 0.001. This negative relationship means that tracts with higher Black population shares tend to have lower median income in this dataset.

This is important for my project because it connects race, housing, and income together. The data is showing that racial composition and economic conditions are not separate issues in Atlanta.


# 2. Hypothesis Testing

For the formal testing section, I used multiple statistical tests because my capstone question has more than one part.

I used:
- T-tests to compare near vs farther BeltLine areas
- ANOVA to compare multiple distance groups
- Pearson correlation to test linear relationships
- OLS regression to model rent using several predictors


## T-Test: Rent Near the BeltLine vs Farther Away

H0: There is no difference in median gross rent between tracts within 1 mile of the BeltLine and tracts farther than 1 mile.

H1: There is a difference in median gross rent between tracts within 1 mile of the BeltLine and tracts farther than 1 mile.


In [ ]:
# t-test for 2020 rent within 1 mile vs farther than 1 mile

df20 = df[df['year'] == 2020].copy()

near_1_mile = df20[df20['centroid_distance_miles'] <= 1]['median_gross_rent'].dropna()
far_1_mile = df20[df20['centroid_distance_miles'] > 1]['median_gross_rent'].dropna()

t_stat, p_value = ttest_ind(near_1_mile, far_1_mile, equal_var=False)

print('near 1 mile mean rent:', near_1_mile.mean())
print('farther than 1 mile mean rent:', far_1_mile.mean())
print('t-statistic:', t_stat)
print('p-value:', p_value)


## T-Test Interpretation

For 2020, the average median rent within 1 mile of the BeltLine is $1,273. The average median rent farther than 1 mile is $1,331.

The p-value is 0.223. Since this is greater than 0.05, this specific 2020 rent comparison is not statistically significant.

This is still useful because it shows that the relationship is more complicated than simply saying every tract closer to the BeltLine has higher rent in 2020. The broader pattern matters more when looking across years and distance groups.


## T-Test: Home Value Near the BeltLine vs Farther Away

H0: There is no difference in median home value between tracts within 1 mile of the BeltLine and tracts farther than 1 mile.

H1: There is a difference in median home value between tracts within 1 mile of the BeltLine and tracts farther than 1 mile.


In [ ]:
# t-test for 2020 home value within 1 mile vs farther than 1 mile

near_home = df20[df20['centroid_distance_miles'] <= 1]['median_home_value'].dropna()
far_home = df20[df20['centroid_distance_miles'] > 1]['median_home_value'].dropna()

t_home, p_home = ttest_ind(near_home, far_home, equal_var=False)

print('near 1 mile mean home value:', near_home.mean())
print('farther than 1 mile mean home value:', far_home.mean())
print('t-statistic:', t_home)
print('p-value:', p_home)


## Home Value T-Test Interpretation

For 2020, the average median home value within 1 mile of the BeltLine is $364,377. The average median home value farther than 1 mile is $308,875.

The p-value is 0.037. Since this is below 0.05, the difference is statistically significant.

This supports my larger argument because home values near the BeltLine are meaningfully higher. Even though rent alone was not significant in that specific test, home value shows a clearer relationship with BeltLine proximity.


## ANOVA: Percent Black by Distance Group

H0: The average percent Black is the same across all BeltLine distance groups.

H1: At least one distance group has a different average percent Black.


In [ ]:
# anova test for pct black across distance groups

anova_groups_black = [
    group['pct_black'].dropna().values
    for name, group in df.groupby('distance_group')
]

f_stat_black, p_anova_black = f_oneway(*anova_groups_black)

print('F-statistic:', f_stat_black)
print('p-value:', p_anova_black)


## ANOVA Interpretation: Percent Black

The ANOVA test for percent Black across distance groups produced an F-statistic of 2.273 and a p-value of 0.035.

Since the p-value is below 0.05, there is a statistically significant difference in average percent Black across the BeltLine distance groups.

This matters because it supports the idea that racial composition is not evenly distributed by distance. In other words, where a tract is located in relation to the BeltLine is connected to differences in racial makeup.


## ANOVA: Median Gross Rent by Distance Group

H0: Median gross rent is the same across all BeltLine distance groups.

H1: At least one BeltLine distance group has a different average median gross rent.


In [ ]:
# anova test for rent across distance groups

anova_groups_rent = [
    group['median_gross_rent'].dropna().values
    for name, group in df.groupby('distance_group')
]

f_stat_rent, p_anova_rent = f_oneway(*anova_groups_rent)

print('F-statistic:', f_stat_rent)
print('p-value:', p_anova_rent)


## ANOVA Interpretation: Rent

The ANOVA test for rent across distance groups produced an F-statistic of 15.151 and a p-value of < 0.001.

Because the p-value is below 0.05, rent differs significantly across distance groups.

This is one of the strongest pieces of statistical evidence in this notebook. It shows that rent patterns are not randomly distributed across space. Distance from the BeltLine is connected to differences in rent levels.


## ANOVA: 2020 Rent Only

I also tested 2020 rent by distance group because 2020 is the most recent time period in this dataset.


In [ ]:
# anova test for 2020 rent across distance groups

anova_groups_rent_2020 = [
    group['median_gross_rent'].dropna().values
    for name, group in df20.groupby('distance_group')
]

f_stat_rent_2020, p_anova_rent_2020 = f_oneway(*anova_groups_rent_2020)

print('F-statistic:', f_stat_rent_2020)
print('p-value:', p_anova_rent_2020)


## 2020 Rent ANOVA Interpretation

For 2020 only, the ANOVA test gave an F-statistic of 3.298 and a p-value of 0.003.

This is statistically significant, which means rent still differs across BeltLine distance groups in the most recent year.

This helps strengthen the argument because the pattern is not just historical. It is still showing up in the current period of the data.


## Pearson Correlation Tests

Pearson correlation helps test whether two numeric variables have a statistically significant linear relationship.


In [ ]:
# pearson correlation between income and rent

pearson_income_rent = pearsonr(
    df[['median_income', 'median_gross_rent']].dropna()['median_income'],
    df[['median_income', 'median_gross_rent']].dropna()['median_gross_rent']
)

pearson_income_rent


In [ ]:
# pearson correlation between pct black and median income

pearson_black_income = pearsonr(
    df[['pct_black', 'median_income']].dropna()['pct_black'],
    df[['pct_black', 'median_income']].dropna()['median_income']
)

pearson_black_income


## Pearson Correlation Interpretation

The correlation between median income and rent is 0.731, with a p-value of < 0.001. This means the relationship is statistically significant.

The correlation between percent Black and median income is -0.691, with a p-value of < 0.001. This is also statistically significant.

For my capstone, this is important because it shows that race and income are connected in the dataset. Higher Black population share is associated with lower median income, while higher income is associated with higher rent.


# OLS Regression

I used OLS regression to see how much rent can be explained by income, race, poverty, and distance from the BeltLine.

The dependent variable is:

`median_gross_rent`

The independent variables are:
- `median_income`
- `pct_black`
- `poverty_rate`
- `centroid_distance_miles`


In [ ]:
# prepare data for ols regression

model_df = df[
    ['median_gross_rent', 'median_income', 'pct_black',
     'poverty_rate', 'centroid_distance_miles']
].dropna()

X = model_df[
    ['median_income', 'pct_black', 'poverty_rate',
     'centroid_distance_miles']
]

y = model_df['median_gross_rent']

# add constant for regression
X = sm.add_constant(X)

# fit model
ols_model = sm.OLS(y, X).fit()

ols_model.summary()


## OLS Regression Interpretation

The OLS model has an R-squared value of 0.565. This means the model explains about 56.5% of the variation in median gross rent.

Median income is statistically significant with a coefficient of 0.0072. This means that as median income increases, median rent also tends to increase.

Distance from the BeltLine is also statistically significant with a coefficient of 15.10 and a p-value of < 0.001. In this model, distance has a positive coefficient, which means rent is not only shaped by closeness to the BeltLine. Other neighborhood-level patterns are also affecting rent.

Percent Black has a coefficient of -64.67, but the p-value is 0.091. This means it is not statistically significant at the 0.05 level in this specific model.

The main takeaway is that rent is strongly connected to income and spatial factors, but the relationship is complex. This is why the project needs both maps and statistical analysis.


# 3. Pre-ML Data Diagnosis & Remediation

Before machine learning, I need to check the data for problems that could hurt model performance.

The main issues I checked are:
- missing values
- outliers
- multicollinearity
- class imbalance


## Missing Values

In [ ]:
# check missing values

missing_values = df.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]


## Missing Values Interpretation

The variables with missing values are mainly housing and poverty-related fields. This makes sense because census/ACS data can have missing or suppressed values in some tracts.

I would not automatically replace all missing values with the mean because that could make the data look cleaner than it really is. For this analysis, I use `dropna()` inside each test or model so each section only removes rows needed for that specific analysis.

This is better than dropping everything upfront because it keeps more data available for sections that do not need the missing variable.


## Outlier Review

In [ ]:
# look at possible outliers using boxplots

outlier_vars = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'poverty_rate'
]

for col in outlier_vars:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[col])
    plt.title(f'Outlier Check: {col}')
    plt.show()


## Outlier Interpretation

There are outliers in variables like income, home value, rent, and poverty rate. I decided to keep these outliers because they represent real neighborhood differences.

For example, a high-income tract is not automatically a data error. It may reflect the uneven development patterns that this project is trying to study.

If I were building a final predictive model, I may test transformations or robust models, but I would not delete outliers without a strong reason.


## Multicollinearity Check

Multicollinearity happens when independent variables are too strongly related to each other. This can make models unstable and make it harder to understand which variable is really driving the outcome.


In [ ]:
# vif check for model features

features = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'pct_white',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

vif_df = df[features].replace([np.inf, -np.inf], np.nan).dropna()

vif_results = pd.DataFrame()
vif_results['feature'] = features
vif_results['VIF'] = [
    variance_inflation_factor(vif_df.values, i)
    for i in range(vif_df.shape[1])
]

vif_results


## Multicollinearity Interpretation

The VIF results show that some variables are too closely related. Median income has a VIF of 23.67, median home value has a VIF of 11.62, and median gross rent has a VIF of 18.19.

This is expected because income, home value, and rent are all connected. But for machine learning, including all of them at once could cause multicollinearity problems, especially in regression models.

To fix this, I would not use median gross rent as both a predictor and an outcome. If rent is my target variable, then I would remove it from the predictor list. I would also consider removing either median income or median home value depending on what I am trying to predict.


In [ ]:
# updated vif after removing home value and rent from predictors

fixed_features = [
    'median_income',
    'pct_black',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

fixed_vif_df = df[fixed_features].replace([np.inf, -np.inf], np.nan).dropna()

fixed_vif_results = pd.DataFrame()
fixed_vif_results['feature'] = fixed_features
fixed_vif_results['VIF'] = [
    variance_inflation_factor(fixed_vif_df.values, i)
    for i in range(fixed_vif_df.shape[1])
]

fixed_vif_results


## Multicollinearity Fix

After removing median home value and median gross rent from the predictor set, the VIF values improved a lot. Median income dropped to a VIF of 2.86, and centroid distance dropped to 2.95.

This is a cleaner feature set for future modeling because the variables are not overlapping as much.

For future machine learning, my cleaned predictor set would likely include:
- median income
- percent Black
- percent renter
- poverty rate
- distance from the BeltLine

This keeps the model connected to my research question without overloading it with duplicate housing cost variables.


## Class Imbalance Check

For classification, I need to check whether the target variable is balanced.

Here, I use `beltline_three_miles` as a possible classification target. This variable identifies whether a tract is within 3 miles of the BeltLine.


In [ ]:
# class imbalance check

df['beltline_three_miles'].value_counts()


In [ ]:
# class percentages

df['beltline_three_miles'].value_counts(normalize=True) * 100


## Class Imbalance Interpretation

For `beltline_three_miles`, 299 tracts are within 3 miles of the BeltLine and 555 tracts are not within 3 miles.

That means about 35.0% of the data is in the within-3-miles class, while about 65.0% is outside of that range.

This is somewhat imbalanced, but not extreme. If I use this as a classification target, I would use stratified train-test split and possibly class weights so the model does not mostly learn the larger class.


In [ ]:
# example of stratified split for future classification

features_for_ml = [
    'median_income',
    'pct_black',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

target = 'beltline_three_miles'

ml_df = df[features_for_ml + [target]].dropna()

X = ml_df[features_for_ml]
y = ml_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('training target balance:')
print(y_train.value_counts(normalize=True) * 100)

print('\ntesting target balance:')
print(y_test.value_counts(normalize=True) * 100)


In [ ]:
# example of class weights if needed

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

class_weight_dict


## Class Imbalance Fix

To handle class imbalance, I would use a stratified train-test split. This keeps the class proportions similar in the training and testing data.

If the model still performs poorly on the smaller class, I would use class weights. I would try class weights before SMOTE because this dataset is spatial and census-based, so creating synthetic observations could distort the geography.


# Final Pre-ML Dataset

This section creates a cleaner version of the dataset that could be used for modeling.


In [ ]:
# create cleaned modeling dataset

model_features = [
    'median_income',
    'pct_black',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

model_target = 'beltline_three_miles'

model_ready = df[model_features + [model_target]].dropna()

model_ready.head()


In [ ]:
# check final shape

model_ready.shape


In [ ]:
# optional: scale features for models that need scaling

scaler = StandardScaler()

scaled_features = scaler.fit_transform(model_ready[model_features])

scaled_df = pd.DataFrame(
    scaled_features,
    columns=model_features
)

scaled_df[model_target] = model_ready[model_target].values

scaled_df.head()


# Final Summary

This analysis shows that the BeltLine story is not just about redevelopment. It is also about cost, race, and space.

The strongest findings are:
- Within 3 miles of the BeltLine, median income increased sharply from 2000 to 2020.
- Within 3 miles, home values and rent also increased heavily.
- The Black population share near the BeltLine declined, while the White population share increased.
- ANOVA showed statistically significant differences in percent Black across distance groups.
- ANOVA also showed statistically significant differences in rent across distance groups.
- The regression model explained a meaningful amount of variation in rent.
- Multicollinearity was present, especially between income, home value, and rent, so the feature set needs to be cleaned before machine learning.
- The possible classification target is somewhat imbalanced, so stratified splitting and class weights should be considered.

Overall, the data supports my capstone argument: redevelopment around the BeltLine is connected to rising housing costs and demographic change. The pattern is not simple, but it is clear enough to show that proximity, race, and housing costs should be studied together.


# GitHub Notes

To upload this notebook to GitHub:

1. Go to your GitHub repository.
2. Click **Add file**.
3. Click **Upload files**.
4. Upload this `.ipynb` file and the `atlanta_spatial_final.csv` file.
5. Commit the changes.
6. Open the notebook on GitHub and copy that link for the assignment.

Make sure the CSV file is in the same folder as the notebook, because the notebook reads it using:

```python
df = pd.read_csv('atlanta_spatial_final.csv')
```
